In [ ]:
#Importing necessary libraries:

import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import re
from pygam import LinearGAM, s, f, l
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
import scipy.stats as stats




In [ ]:
#Loading the data into dataframe

df_hba1c = pd.read_stata("data/Hb1Ac_CIMT.dta")


print(df_hba1c.shape)
list(df_hba1c.columns)

In [ ]:
# T2D excluded
x = pd.to_numeric(df_hba1c["hba1c_nondiabetic"], errors="coerce").dropna()

print("HbA1c T2D excluded N:", len(x))
print("HbA1c T2D excluded mean:", x.mean())
print("HbA1c T2D excluded SD:", x.std())


# T2D included
y = pd.to_numeric(df_hba1c["hba1c_clean"], errors="coerce").dropna()

print("HbA1c T2D included N:", len(y))
print("HbA1c T2D included mean:", y.mean())
print("HbA1c T2D included SD:", y.std())

In [ ]:
#Basic check on hb1ac variable

hba1c_cols = [
    'hba1c_0'
]

#Summary stats
# df_hba1c[hba1c_cols].describe().T

#Missingnes
# df_hba1c[hba1c_cols].isna().sum().sort_values(ascending=False)

# #Skewness
# df_hba1c[hba1c_cols].skew().sort_values(ascending=False)

df_hba1c[hba1c_cols].hist(bins=50, figsize=(12, 8))

In [ ]:
#checking the extreme values


#checking number of values within thresholds
for cutoff in [48, 60, 80, 100, 150, 200]:
    print(cutoff, (df_hba1c["hba1c_0"] > cutoff).sum())

In [ ]:
#final hb1ac variable
df_hba1c["hba1c_clean"] = df_hba1c["hba1c_0"]

df_hba1c.loc[
    (df_hba1c["hba1c_clean"] < 15) |
    (df_hba1c["hba1c_clean"] > 150),
    "hba1c_clean"
] = np.nan

#checking quantiles
df_hba1c["hba1c_0"].quantile([
    0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9995, 0.9999
])

In [ ]:
#creating a column for HBa1c values excluding ones greater than 48

df_hba1c["hba1c_nondiabetic"] = df_hba1c["hba1c_clean"]

df_hba1c.loc[
    df_hba1c["hba1c_nondiabetic"] >= 48,
    "hba1c_nondiabetic"
] = np.nan

print(df_hba1c["hba1c_clean"].describe())
print(df_hba1c["hba1c_clean"].skew())

print(df_hba1c["hba1c_nondiabetic"].describe())
print(df_hba1c["hba1c_nondiabetic"].skew())

In [ ]:
hba1c_cols = ["hba1c_nondiabetic","hba1c_clean"]
df_hba1c[hba1c_cols].hist(bins=50, figsize=(12, 8))

In [ ]:
#Loading Prs data
df_prs = pd.read_stata("data/UKB 71392 CMD Project.dta")

#checking overlap for clean

hba1c_clean_available_df = df_hba1c.dropna(subset=["hba1c_clean"]).copy()

overlap_hba1c_clean = hba1c_clean_available_df["n_eid"].isin(df_prs["n_eid"]).sum()

print("HbA1c clean available:", hba1c_clean_available_df.shape[0])
print("Overlap with PRS dataset:", overlap_hba1c_clean)

#checking overlap for nondiabetic

hba1c_nd_available_df = df_hba1c.dropna(subset=["hba1c_nondiabetic"]).copy()

overlap_hba1c_nd = hba1c_nd_available_df["n_eid"].isin(df_prs["n_eid"]).sum()

print("HbA1c nondiabetic available:", hba1c_nd_available_df.shape[0])
print("Overlap with PRS dataset:", overlap_hba1c_nd)

In [ ]:
#merging Hba1c and PRS data

df_hba1c_merge = df_hba1c[
    ["n_eid", "hba1c_clean", "hba1c_nondiabetic"]
].copy()

df_hba1c_model = df_prs.merge(
    df_hba1c_merge,
    on="n_eid",
    how="inner"
)

print(df_hba1c_model.shape)

print(df_hba1c_model[["hba1c_clean", "hba1c_nondiabetic"]].notna().sum())

In [ ]:
#checking if all columns are in there:

needed_cols = [
    "prs_adhd_std",
    "hba1c_clean",
    "hba1c_nondiabetic",
    "chip",
    "PGC1", "PGC2", "PGC3", "PGC4",
    "PGC5", "PGC6", "PGC7", "PGC8",
    "age_0_0",
    "sex_plink",
    "townsend",
    "cursmoke_0",
    "curr_alcohol_0",
    "n_21001_0_0"
]

missing_cols = [col for col in needed_cols if col not in df_hba1c_model.columns]
print("Missing columns:", missing_cols)

print("Duplicate n_eid:", df_hba1c_model["n_eid"].duplicated().sum())

In [ ]:
# Cleaning the HbA1c modelling dataset(nondiabetic)

# Fixing the sex column and turning it binary
# 0 = male, 1 = female
df_hba1c_model["sex"] = df_hba1c_model["sex_plink"].map({1: 0, 2: 1})

# Main variables needed for the fully adjusted HbA1c models
main_vars_hba1c_nd = [
    "prs_adhd_std",
    "hba1c_nondiabetic",
    "n_21001_0_0",
    "age_0_0",
    "sex",
    "townsend",
    "cursmoke_0",
    "curr_alcohol_0",
    "PGC1", "PGC2", "PGC3", "PGC4",
    "PGC5", "PGC6", "PGC7", "PGC8",
    "chip"
]

df_hba1c_nd_model = df_hba1c_model.dropna(subset=main_vars_hba1c_nd).copy()

print(df_hba1c_nd_model.shape)
list(df_hba1c_nd_model.columns)

In [ ]:
# Cleaning the HbA1c modelling dataset(nondiabetic)
main_vars_hba1c_clean = [
    "prs_adhd_std",
    "hba1c_clean",
    "n_21001_0_0",
    "age_0_0",
    "sex",
    "townsend",
    "cursmoke_0",
    "curr_alcohol_0",
    "PGC1", "PGC2", "PGC3", "PGC4",
    "PGC5", "PGC6", "PGC7", "PGC8",
    "chip"
]

df_hba1c_clean_model = df_hba1c_model.dropna(subset=main_vars_hba1c_clean).copy()

print(df_hba1c_clean_model.shape)
list(df_hba1c_clean_model.columns)

In [ ]:
# Tertile analysis: HbA1c, T2D excluded

df_hba1c_nd_tertile = df_hba1c_nd_model[
    ['prs_adhd_std', 'hba1c_nondiabetic']
].dropna().copy()

# Create ADHD PRS tertiles
df_hba1c_nd_tertile['prs_tertile'] = pd.qcut(
    df_hba1c_nd_tertile['prs_adhd_std'],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Summary statistics
hba1c_nd_tertile_summary = (
    df_hba1c_nd_tertile
    .groupby('prs_tertile', observed=False)['hba1c_nondiabetic']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)

print(hba1c_nd_tertile_summary)

# Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(
    x='prs_tertile',
    y='hba1c_nondiabetic',
    data=df_hba1c_nd_tertile,
    palette='Set2'
)

plt.title('HbA1c across ADHD PRS Tertiles: T2D Excluded')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('HbA1c (mmol/mol)')
plt.tight_layout()
plt.show()

In [ ]:
# Tertile analysis: HbA1c, T2D included

df_hba1c_clean_tertile = df_hba1c_clean_model[
    ['prs_adhd_std', 'hba1c_clean']
].dropna().copy()

# Create ADHD PRS tertiles
df_hba1c_clean_tertile['prs_tertile'] = pd.qcut(
    df_hba1c_clean_tertile['prs_adhd_std'],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Summary statistics
hba1c_clean_tertile_summary = (
    df_hba1c_clean_tertile
    .groupby('prs_tertile', observed=False)['hba1c_clean']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
)

print(hba1c_clean_tertile_summary)

# Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(
    x='prs_tertile',
    y='hba1c_clean',
    data=df_hba1c_clean_tertile,
    palette='Set2'
)

plt.title('HbA1c across ADHD PRS Tertiles: T2D Included')
plt.xlabel('ADHD PRS Tertile')
plt.ylabel('HbA1c (mmol/mol)')
plt.tight_layout()
plt.show()

In [ ]:
#scaling Age and townsend

from sklearn.preprocessing import StandardScaler

scale_cols = ["age_0_0", "townsend"]

scaler_nd = StandardScaler()
df_hba1c_nd_model[scale_cols] = scaler_nd.fit_transform(
    df_hba1c_nd_model[scale_cols]
)

scaler_clean = StandardScaler()
df_hba1c_clean_model[scale_cols] = scaler_clean.fit_transform(
    df_hba1c_clean_model[scale_cols]
)

In [ ]:
# Model 1: Non-diabetic HbA1c ~ ADHD PRS + genetic covariates

# Filter and select relevant columns
cols = [
    'prs_adhd_std',
    'hba1c_nondiabetic',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model1 = df_hba1c_nd_model[cols].dropna().copy()

# Define predictors (X) and outcome (y)
X = df_model1.drop(columns='hba1c_nondiabetic')

# Convert chip into dummy variables
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model1.loc[X.index, 'hba1c_nondiabetic']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model1_hba1c_nd = sm.OLS(y, X).fit(cov_type="HC3")

print(model1_hba1c_nd.summary())

# Predict and attach predictions back to dataframe
df_model1 = df_model1.loc[X.index].copy()
df_model1['hba1c_predicted'] = model1_hba1c_nd.predict(X)

# Bin the PRS for smoother plotting
df_model1['prs_bin'] = pd.cut(df_model1['prs_adhd_std'], bins=100)

binned = (
    df_model1
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_nondiabetic',
    data=df_model1,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

# Smoothed OLS fit
sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 1: ADHD PRS + Genetic Covariates vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Non-diabetic HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 2: Non-diabetic HbA1c ~ ADHD PRS + genetic covariates + age + sex

cols = [
    'prs_adhd_std',
    'hba1c_nondiabetic',
    'age_0_0',
    'sex',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model2 = df_hba1c_nd_model[cols].dropna().copy()

X = df_model2.drop(columns='hba1c_nondiabetic')

X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

y = df_model2.loc[X.index, 'hba1c_nondiabetic']

X = sm.add_constant(X)

model2_hba1c_nd = sm.OLS(y, X).fit(cov_type="HC3")

print(model2_hba1c_nd.summary())

df_model2 = df_model2.loc[X.index].copy()
df_model2['hba1c_predicted'] = model2_hba1c_nd.predict(X)

df_model2['prs_bin'] = pd.cut(df_model2['prs_adhd_std'], bins=100)

binned = (
    df_model2
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_nondiabetic',
    data=df_model2,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 2: ADHD PRS + Age + Sex vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Non-diabetic HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 3: Non-diabetic HbA1c ~ ADHD PRS + genetic covariates + age + sex + lifestyle + SES

cols = [
    'prs_adhd_std',
    'hba1c_nondiabetic',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model3 = df_hba1c_nd_model[cols].dropna().copy()

# Define predictors and outcome
X = df_model3.drop(columns='hba1c_nondiabetic')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model3.loc[X.index, 'hba1c_nondiabetic']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model3_hba1c_nd = sm.OLS(y, X).fit(cov_type="HC3")

print(model3_hba1c_nd.summary())

# Predict and attach predictions back to dataframe
df_model3 = df_model3.loc[X.index].copy()
df_model3['hba1c_predicted'] = model3_hba1c_nd.predict(X)

# Bin the PRS for smoother plotting
df_model3['prs_bin'] = pd.cut(df_model3['prs_adhd_std'], bins=100)

binned = (
    df_model3
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_nondiabetic',
    data=df_model3,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 3: ADHD PRS + Lifestyle Covariates + SES vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Non-diabetic HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 4: Non-diabetic HbA1c ~ ADHD PRS + genetic covariates + age + sex + lifestyle + SES + BMI

cols = [
    'prs_adhd_std',
    'hba1c_nondiabetic',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model4 = df_hba1c_nd_model[cols].dropna().copy()

# Define predictors and outcome
X = df_model4.drop(columns='hba1c_nondiabetic')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model4.loc[X.index, 'hba1c_nondiabetic']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model4_hba1c_nd = sm.OLS(y, X).fit(cov_type="HC3")

print(model4_hba1c_nd.summary())

# Predict and attach predictions back to dataframe
df_model4 = df_model4.loc[X.index].copy()
df_model4['hba1c_predicted'] = model4_hba1c_nd.predict(X)

# Bin the PRS for smoother plotting
df_model4['prs_bin'] = pd.cut(df_model4['prs_adhd_std'], bins=100)

binned = (
    df_model4
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_nondiabetic',
    data=df_model4,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 4: ADHD PRS + Lifestyle Covariates + BMI vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Non-diabetic HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

End of OLS for ND

In [ ]:
# GAM 1: Non-diabetic HbA1c ~ ADHD PRS + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam1_cols = [
    'hba1c_nondiabetic',
    'prs_adhd_std',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam1 = df_hba1c_nd_model[gam1_cols].dropna().copy()

# Convert chip into numeric category codes for pyGAM
df_gam1['chip'] = df_gam1['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam1 = [
    'prs_adhd_std',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X1_gam = df_gam1[feature_cols_gam1].values
y1_gam = df_gam1['hba1c_nondiabetic'].values

# Fit GAM Model 1
# s(0) = spline for ADHD PRS
# f(1) = factor term for chip
# l(2) to l(9) = linear genetic PC covariates

gam1_hba1c_nd = LinearGAM(
    s(0) +
    f(1) +
    l(2) + l(3) + l(4) + l(5) +
    l(6) + l(7) + l(8) + l(9)
).fit(X1_gam, y1_gam)

# Print summary
gam1_hba1c_nd.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam1_hba1c_nd.generate_X_grid(term=term)
pdep, confi = gam1_hba1c_nd.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 1: ADHD PRS + Genetic Covariates vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on non-diabetic HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 2: Non-diabetic HbA1c ~ ADHD PRS + age + sex + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam2_cols = [
    'hba1c_nondiabetic',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam2 = df_hba1c_nd_model[gam2_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam2['sex'] = df_gam2['sex'].astype('category').cat.codes
df_gam2['chip'] = df_gam2['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam2 = [
    'prs_adhd_std',  # 0
    'age_0_0',       # 1
    'sex',           # 2
    'chip',          # 3
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X2_gam = df_gam2[feature_cols_gam2].values
y2_gam = df_gam2['hba1c_nondiabetic'].values

# Fit GAM Model 2
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# f(3) = chip factor
# l(4) to l(11) = linear genetic PC covariates

gam2_hba1c_nd = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    f(3) +
    l(4) + l(5) + l(6) + l(7) +
    l(8) + l(9) + l(10) + l(11)
).fit(X2_gam, y2_gam)

# Print summary
gam2_hba1c_nd.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam2_hba1c_nd.generate_X_grid(term=term)
pdep, confi = gam2_hba1c_nd.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 2: ADHD PRS + Age + Sex vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on non-diabetic HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 3: Non-diabetic HbA1c ~ ADHD PRS + age + sex + lifestyle + SES + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam3_cols = [
    'hba1c_nondiabetic',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam3 = df_hba1c_nd_model[gam3_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam3['sex'] = df_gam3['sex'].astype('category').cat.codes
df_gam3['cursmoke_0'] = df_gam3['cursmoke_0'].astype('category').cat.codes
df_gam3['curr_alcohol_0'] = df_gam3['curr_alcohol_0'].astype('category').cat.codes
df_gam3['chip'] = df_gam3['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam3 = [
    'prs_adhd_std',      # 0
    'age_0_0',           # 1
    'sex',               # 2
    'townsend',          # 3
    'cursmoke_0',        # 4
    'curr_alcohol_0',    # 5
    'chip',              # 6
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X3_gam = df_gam3[feature_cols_gam3].values
y3_gam = df_gam3['hba1c_nondiabetic'].values

# Fit GAM Model 3
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# s(3) = Townsend spline
# f(4) = smoking factor
# f(5) = alcohol factor
# f(6) = chip factor
# l(7) to l(14) = linear genetic PC covariates

gam3_hba1c_nd = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    f(6) +
    l(7) + l(8) + l(9) + l(10) +
    l(11) + l(12) + l(13) + l(14)
).fit(X3_gam, y3_gam)

# Print summary
gam3_hba1c_nd.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam3_hba1c_nd.generate_X_grid(term=term)
pdep, confi = gam3_hba1c_nd.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 3: ADHD PRS + Lifestyle + SES vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on non-diabetic HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 4: Non-diabetic HbA1c ~ ADHD PRS + age + sex + lifestyle + SES + BMI + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam4_cols = [
    'hba1c_nondiabetic',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam4 = df_hba1c_nd_model[gam4_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam4['sex'] = df_gam4['sex'].astype('category').cat.codes
df_gam4['cursmoke_0'] = df_gam4['cursmoke_0'].astype('category').cat.codes
df_gam4['curr_alcohol_0'] = df_gam4['curr_alcohol_0'].astype('category').cat.codes
df_gam4['chip'] = df_gam4['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam4 = [
    'prs_adhd_std',      # 0
    'age_0_0',           # 1
    'sex',               # 2
    'townsend',          # 3
    'cursmoke_0',        # 4
    'curr_alcohol_0',    # 5
    'n_21001_0_0',       # 6 BMI
    'chip',              # 7
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X4_gam = df_gam4[feature_cols_gam4].values
y4_gam = df_gam4['hba1c_nondiabetic'].values

# Fit GAM Model 4
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# s(3) = Townsend spline
# f(4) = smoking factor
# f(5) = alcohol factor
# s(6) = BMI spline
# f(7) = chip factor
# l(8) to l(15) = linear genetic PC covariates

gam4_hba1c_nd = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    s(6) +
    f(7) +
    l(8) + l(9) + l(10) + l(11) +
    l(12) + l(13) + l(14) + l(15)
).fit(X4_gam, y4_gam)

# Print summary
gam4_hba1c_nd.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam4_hba1c_nd.generate_X_grid(term=term)
pdep, confi = gam4_hba1c_nd.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 4: ADHD PRS + Lifestyle + SES + BMI vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on non-diabetic HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 4: Non-diabetic HbA1c ~ ADHD PRS + age + sex + lifestyle + SES + BMI + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam4_cols = [
    'hba1c_nondiabetic',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam4 = df_hba1c_nd_model[gam4_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam4['sex'] = df_gam4['sex'].astype('category').cat.codes
df_gam4['cursmoke_0'] = df_gam4['cursmoke_0'].astype('category').cat.codes
df_gam4['curr_alcohol_0'] = df_gam4['curr_alcohol_0'].astype('category').cat.codes
df_gam4['chip'] = df_gam4['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam4 = [
    'prs_adhd_std',      # 0
    'age_0_0',           # 1
    'sex',               # 2
    'townsend',          # 3
    'cursmoke_0',        # 4
    'curr_alcohol_0',    # 5
    'n_21001_0_0',       # 6 BMI
    'chip',              # 7
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X4_gam = df_gam4[feature_cols_gam4].values
y4_gam = df_gam4['hba1c_nondiabetic'].values

# Fit GAM Model 4
# l(0) = ADHD PRS linear term
# s(1) = age spline
# f(2) = sex factor
# s(3) = Townsend spline
# f(4) = smoking factor
# f(5) = alcohol factor
# s(6) = BMI spline
# f(7) = chip factor
# l(8) to l(15) = linear genetic PC covariates

gam4_hba1c_nd_linear_prs = LinearGAM(
    l(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    s(6) +
    f(7) +
    l(8) + l(9) + l(10) + l(11) +
    l(12) + l(13) + l(14) + l(15)
).fit(X4_gam, y4_gam)

# Print summary
gam4_hba1c_nd_linear_prs.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam4_hba1c_nd_linear_prs.generate_X_grid(term=term)
pdep, confi = gam4_hba1c_nd_linear_prs.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS linear term'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 4: Linear ADHD PRS + Lifestyle + SES + BMI vs Non-diabetic HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on non-diabetic HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

END OF NONDIABETIC ANALYSIS

In [ ]:
# Model 1: Clean HbA1c ~ ADHD PRS + genetic covariates

cols = [
    'prs_adhd_std',
    'hba1c_clean',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model1_clean = df_hba1c_clean_model[cols].dropna().copy()

X = df_model1_clean.drop(columns='hba1c_clean')

X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

y = df_model1_clean.loc[X.index, 'hba1c_clean']

X = sm.add_constant(X)
model1_hba1c_clean = sm.OLS(y, X).fit(cov_type="HC3")

print(model1_hba1c_clean.summary())

df_model1_clean = df_model1_clean.loc[X.index].copy()
df_model1_clean['hba1c_predicted'] = model1_hba1c_clean.predict(X)

df_model1_clean['prs_bin'] = pd.cut(df_model1_clean['prs_adhd_std'], bins=100)

binned = (
    df_model1_clean
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_clean',
    data=df_model1_clean,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 1: ADHD PRS + Genetic Covariates vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Clean HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 2: Clean HbA1c ~ ADHD PRS + genetic covariates + age + sex

cols = [
    'prs_adhd_std',
    'hba1c_clean',
    'age_0_0',
    'sex',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model2_clean = df_hba1c_clean_model[cols].dropna().copy()

# Define predictors and outcome
X = df_model2_clean.drop(columns='hba1c_clean')

# Convert chip into dummy variables
X = pd.get_dummies(X, columns=['chip'], drop_first=True, dtype=float)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model2_clean.loc[X.index, 'hba1c_clean']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model2_hba1c_clean = sm.OLS(y, X).fit(cov_type="HC3")

print(model2_hba1c_clean.summary())

# Predict and attach predictions back to dataframe
df_model2_clean = df_model2_clean.loc[X.index].copy()
df_model2_clean['hba1c_predicted'] = model2_hba1c_clean.predict(X)

# Bin the PRS for smoother plotting
df_model2_clean['prs_bin'] = pd.cut(df_model2_clean['prs_adhd_std'], bins=100)

binned = (
    df_model2_clean
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_clean',
    data=df_model2_clean,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 2: ADHD PRS + Age + Sex vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Clean HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 3: Clean HbA1c ~ ADHD PRS + genetic covariates + age + sex + lifestyle + SES

cols = [
    'prs_adhd_std',
    'hba1c_clean',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model3_clean = df_hba1c_clean_model[cols].dropna().copy()

# Define predictors and outcome
X = df_model3_clean.drop(columns='hba1c_clean')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model3_clean.loc[X.index, 'hba1c_clean']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model3_hba1c_clean = sm.OLS(y, X).fit(cov_type="HC3")

print(model3_hba1c_clean.summary())

# Predict and attach predictions back to dataframe
df_model3_clean = df_model3_clean.loc[X.index].copy()
df_model3_clean['hba1c_predicted'] = model3_hba1c_clean.predict(X)

# Bin the PRS for smoother plotting
df_model3_clean['prs_bin'] = pd.cut(df_model3_clean['prs_adhd_std'], bins=100)

binned = (
    df_model3_clean
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_clean',
    data=df_model3_clean,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 3: ADHD PRS + Lifestyle Covariates + SES vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Clean HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model 4: Clean HbA1c ~ ADHD PRS + genetic covariates + age + sex + lifestyle + SES + BMI

cols = [
    'prs_adhd_std',
    'hba1c_clean',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_model4_clean = df_hba1c_clean_model[cols].dropna().copy()

# Define predictors and outcome
X = df_model4_clean.drop(columns='hba1c_clean')

# Convert categorical variables into dummy variables
X = pd.get_dummies(
    X,
    columns=['chip', 'cursmoke_0', 'curr_alcohol_0'],
    drop_first=True,
    dtype=float
)

# Ensure all predictors are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()

# Align outcome with cleaned predictors
y = df_model4_clean.loc[X.index, 'hba1c_clean']

# Add intercept and fit OLS model
X = sm.add_constant(X)
model4_hba1c_clean = sm.OLS(y, X).fit(cov_type="HC3")

print(model4_hba1c_clean.summary())

# Predict and attach predictions back to dataframe
df_model4_clean = df_model4_clean.loc[X.index].copy()
df_model4_clean['hba1c_predicted'] = model4_hba1c_clean.predict(X)

# Bin the PRS for smoother plotting
df_model4_clean['prs_bin'] = pd.cut(df_model4_clean['prs_adhd_std'], bins=100)

binned = (
    df_model4_clean
    .groupby('prs_bin', observed=False)[['prs_adhd_std', 'hba1c_predicted']]
    .mean()
    .dropna()
)

# Plot
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x='prs_adhd_std',
    y='hba1c_clean',
    data=df_model4_clean,
    alpha=0.1,
    edgecolor=None,
    label='Raw Data'
)

sns.lineplot(
    x='prs_adhd_std',
    y='hba1c_predicted',
    data=binned,
    color='red',
    label='OLS Fit (Smoothed)'
)

plt.title('Model 4: ADHD PRS + Lifestyle Covariates + BMI vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Clean HbA1c mmol/mol')
plt.legend()
plt.tight_layout()
plt.show()

End of OLS for clean

In [ ]:
# GAM 1: Clean HbA1c ~ ADHD PRS + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam1_cols = [
    'hba1c_clean',
    'prs_adhd_std',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam1_clean = df_hba1c_clean_model[gam1_cols].dropna().copy()

# Convert chip into numeric category codes for pyGAM
df_gam1_clean['chip'] = df_gam1_clean['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam1_clean = [
    'prs_adhd_std',  # 0
    'chip',          # 1
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X1_gam_clean = df_gam1_clean[feature_cols_gam1_clean].values
y1_gam_clean = df_gam1_clean['hba1c_clean'].values

# Fit GAM Model 1
# s(0) = ADHD PRS spline
# f(1) = chip factor
# l(2) to l(9) = linear genetic PC covariates

gam1_hba1c_clean = LinearGAM(
    s(0) +
    f(1) +
    l(2) + l(3) + l(4) + l(5) +
    l(6) + l(7) + l(8) + l(9)
).fit(X1_gam_clean, y1_gam_clean)

# Print summary
gam1_hba1c_clean.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam1_hba1c_clean.generate_X_grid(term=term)
pdep, confi = gam1_hba1c_clean.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 1: ADHD PRS + Genetic Covariates vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on clean HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 2: Clean HbA1c ~ ADHD PRS + age + sex + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam2_cols = [
    'hba1c_clean',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam2_clean = df_hba1c_clean_model[gam2_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam2_clean['sex'] = df_gam2_clean['sex'].astype('category').cat.codes
df_gam2_clean['chip'] = df_gam2_clean['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam2_clean = [
    'prs_adhd_std',  # 0
    'age_0_0',       # 1
    'sex',           # 2
    'chip',          # 3
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X2_gam_clean = df_gam2_clean[feature_cols_gam2_clean].values
y2_gam_clean = df_gam2_clean['hba1c_clean'].values

# Fit GAM Model 2
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# f(3) = chip factor
# l(4) to l(11) = linear genetic PC covariates

gam2_hba1c_clean = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    f(3) +
    l(4) + l(5) + l(6) + l(7) +
    l(8) + l(9) + l(10) + l(11)
).fit(X2_gam_clean, y2_gam_clean)

# Print summary
gam2_hba1c_clean.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam2_hba1c_clean.generate_X_grid(term=term)
pdep, confi = gam2_hba1c_clean.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 2: ADHD PRS + Age + Sex vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on clean HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 3: Clean HbA1c ~ ADHD PRS + age + sex + lifestyle + SES + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam3_cols = [
    'hba1c_clean',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam3_clean = df_hba1c_clean_model[gam3_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam3_clean['sex'] = df_gam3_clean['sex'].astype('category').cat.codes
df_gam3_clean['cursmoke_0'] = df_gam3_clean['cursmoke_0'].astype('category').cat.codes
df_gam3_clean['curr_alcohol_0'] = df_gam3_clean['curr_alcohol_0'].astype('category').cat.codes
df_gam3_clean['chip'] = df_gam3_clean['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam3_clean = [
    'prs_adhd_std',      # 0
    'age_0_0',           # 1
    'sex',               # 2
    'townsend',          # 3
    'cursmoke_0',        # 4
    'curr_alcohol_0',    # 5
    'chip',              # 6
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X3_gam_clean = df_gam3_clean[feature_cols_gam3_clean].values
y3_gam_clean = df_gam3_clean['hba1c_clean'].values

# Fit GAM Model 3
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# s(3) = Townsend spline
# f(4) = smoking factor
# f(5) = alcohol factor
# f(6) = chip factor
# l(7) to l(14) = linear genetic PC covariates

gam3_hba1c_clean = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    f(6) +
    l(7) + l(8) + l(9) + l(10) +
    l(11) + l(12) + l(13) + l(14)
).fit(X3_gam_clean, y3_gam_clean)

# Print summary
gam3_hba1c_clean.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam3_hba1c_clean.generate_X_grid(term=term)
pdep, confi = gam3_hba1c_clean.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 3: ADHD PRS + Lifestyle + SES vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on clean HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GAM 4: Clean HbA1c ~ ADHD PRS + age + sex + lifestyle + SES + BMI + genetic covariates

from pygam import LinearGAM, s, l, f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Select relevant columns
gam4_cols = [
    'hba1c_clean',
    'prs_adhd_std',
    'age_0_0',
    'sex',
    'townsend',
    'cursmoke_0',
    'curr_alcohol_0',
    'n_21001_0_0',
    'chip',
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

df_gam4_clean = df_hba1c_clean_model[gam4_cols].dropna().copy()

# Convert categorical variables into numeric category codes for pyGAM
df_gam4_clean['sex'] = df_gam4_clean['sex'].astype('category').cat.codes
df_gam4_clean['cursmoke_0'] = df_gam4_clean['cursmoke_0'].astype('category').cat.codes
df_gam4_clean['curr_alcohol_0'] = df_gam4_clean['curr_alcohol_0'].astype('category').cat.codes
df_gam4_clean['chip'] = df_gam4_clean['chip'].astype('category').cat.codes

# Define feature order
feature_cols_gam4_clean = [
    'prs_adhd_std',      # 0
    'age_0_0',           # 1
    'sex',               # 2
    'townsend',          # 3
    'cursmoke_0',        # 4
    'curr_alcohol_0',    # 5
    'n_21001_0_0',       # 6 BMI
    'chip',              # 7
    'PGC1', 'PGC2', 'PGC3', 'PGC4',
    'PGC5', 'PGC6', 'PGC7', 'PGC8'
]

X4_gam_clean = df_gam4_clean[feature_cols_gam4_clean].values
y4_gam_clean = df_gam4_clean['hba1c_clean'].values

# Fit GAM Model 4
# s(0) = ADHD PRS spline
# s(1) = age spline
# f(2) = sex factor
# s(3) = Townsend spline
# f(4) = smoking factor
# f(5) = alcohol factor
# s(6) = BMI spline
# f(7) = chip factor
# l(8) to l(15) = linear genetic PC covariates

gam4_hba1c_clean = LinearGAM(
    s(0) +
    s(1) +
    f(2) +
    s(3) +
    f(4) +
    f(5) +
    s(6) +
    f(7) +
    l(8) + l(9) + l(10) + l(11) +
    l(12) + l(13) + l(14) + l(15)
).fit(X4_gam_clean, y4_gam_clean)

# Print summary
gam4_hba1c_clean.summary()

# Plot partial dependence for ADHD PRS
term = 0

XX = gam4_hba1c_clean.generate_X_grid(term=term)
pdep, confi = gam4_hba1c_clean.partial_dependence(
    term=term,
    X=XX,
    width=0.95
)

plt.figure(figsize=(8, 6))

plt.plot(
    XX[:, term],
    pdep,
    label='ADHD PRS smooth'
)

plt.fill_between(
    XX[:, term],
    confi[:, 0],
    confi[:, 1],
    alpha=0.2,
    label='95% CI'
)

plt.axhline(0, linestyle='--', linewidth=1)

plt.title('GAM 4: ADHD PRS + Lifestyle + SES + BMI vs Clean HbA1c')
plt.xlabel('Standardized ADHD PRS')
plt.ylabel('Partial effect on clean HbA1c')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# EDIT HERE
# ============================================================

# Main dataframe
# Replace this with your actual dataframe name
main_df = df_hba1c

# PRS column
prs_col = "prs_adhd_std"

# Outcome columns
# Replace these with your exact column names if different
outcome_cols = {
    "BMI": "bmi",
    "WHR": "whr",
    "SBP": "sbp_adjusted",
    "DBP": "dbp_adjusted",
    "cIMT": "cimt",
    "HbA1c": "hba1c_nondiabetic"
}

# ============================================================
# COLOURS
# ============================================================

low_color = "#76B7A5"
medium_color = "#E89A73"
high_color = "#8D9FC3"

tertile_colors = [low_color, medium_color, high_color]

text_color = "#243447"
grid_color = "#E5E7EB"
line_color = "#7C8794"

# ============================================================
# FUNCTION: create outcome-specific tertile dataframe
# ============================================================

def make_tertile_df(df, prs_col, outcome_col):
    temp = df[[prs_col, outcome_col]].dropna().copy()

    temp["prs_tertile"] = pd.qcut(
        temp[prs_col],
        q=3,
        labels=["Low", "Medium", "High"]
    )

    return temp

# ============================================================
# FIGURE SETUP
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

# ============================================================
# LOOP THROUGH OUTCOMES
# ============================================================

for ax, (display_name, outcome_col) in zip(axes, outcome_cols.items()):

    temp = make_tertile_df(main_df, prs_col, outcome_col)

    groups = [
        temp.loc[temp["prs_tertile"] == tertile, outcome_col].values
        for tertile in ["Low", "Medium", "High"]
    ]

    bp = ax.boxplot(
        groups,
        patch_artist=True,
        widths=0.6,
        showfliers=False,   # cleaner for poster
        medianprops=dict(color="#4B5563", linewidth=1.8),
        whiskerprops=dict(color=line_color, linewidth=1.2),
        capprops=dict(color=line_color, linewidth=1.2),
        boxprops=dict(color=line_color, linewidth=1.2)
    )

    # Fill box colours
    for patch, color in zip(bp["boxes"], tertile_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.90)

    # Labels
    ax.set_title(display_name, fontsize=14, fontweight="bold", color=text_color, pad=8)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Low", "Medium", "High"], fontsize=10.5)
    ax.tick_params(axis="y", labelsize=9.5)

    # Grid and spines
    ax.grid(axis="y", color=grid_color, linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CBD5E1")
    ax.spines["bottom"].set_color("#CBD5E1")

    # Add median labels above boxes
    medians = temp.groupby("prs_tertile", observed=False)[outcome_col].median().reindex(["Low", "Medium", "High"])

    for i, median in enumerate(medians, start=1):
        y_min, y_max = ax.get_ylim()
        offset = (y_max - y_min) * 0.03
        ax.text(
            i,
            median + offset,
            f"{median:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            color=text_color,
            fontweight="bold"
        )

# ============================================================
# COMMON TITLE AND LABELS
# ============================================================

fig.suptitle(
    "Cardiometabolic traits across PRS-ADHD tertiles",
    fontsize=18,
    fontweight="bold",
    color=text_color,
    y=0.98
)

fig.text(
    0.5,
    0.04,
    "PRS-ADHD tertile",
    ha="center",
    fontsize=11,
    color=text_color
)

# ============================================================
# LEGEND
# ============================================================

legend_handles = [
    Patch(facecolor=low_color, edgecolor=line_color, label="Low"),
    Patch(facecolor=medium_color, edgecolor=line_color, label="Medium"),
    Patch(facecolor=high_color, edgecolor=line_color, label="High")
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=3,
    frameon=False,
    fontsize=10.5
)

# ============================================================
# FOOTNOTE
# ============================================================

fig.text(
    0.5,
    0.01,
    "Outcome-specific tertiles were created after excluding missing data for each trait. "
    "HbA1c uses the non-diabetic sample.",
    ha="center",
    fontsize=9.5,
    color="#64748B"
)

plt.tight_layout(rect=[0.03, 0.06, 0.99, 0.90])
plt.show()